# Workforce Analytics EDA — BPO Operations
**Author:** Juan Carlos Mejía Soto  
**Data:** Simulated BPO dataset (80 agents, 3 campaigns, 90 days)  
**Goal:** Exploratory analysis of workforce KPIs to support dashboard design and executive reporting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120

CAMPAIGN_COLORS = {
    'Tech Support': '#4C72B0',
    'Billing & Collections': '#DD8452',
    'Customer Retention': '#55A868',
}

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/workforce_bpo_simulated_data.csv', parse_dates=['date'])
df['week'] = df['date'].dt.isocalendar().week.astype(int)
df['week_label'] = 'W' + df['week'].astype(str).str.zfill(2)
df['day_of_week'] = df['date'].dt.day_name()

active = df[df['absences'] == 0].copy()

print(f'Total records: {df.shape[0]:,}')
print(f'Active (non-absent): {active.shape[0]:,}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Agents: {df["agent_id"].nunique()}')
print(f'Campaigns: {df["campaign"].unique().tolist()}')
df.dtypes

## 2. Dataset Overview

In [ ]:
active.describe().round(2)

In [ ]:
# Missing values check
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0])

## 3. Analysis 1 — Adherence Trend by Week

In [ ]:
trend = (
    active.groupby(['week_label', 'campaign'])['adherence_pct']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots()
for campaign, color in CAMPAIGN_COLORS.items():
    sub = trend[trend['campaign'] == campaign]
    ax.plot(sub['week_label'], sub['adherence_pct'], marker='o', label=campaign, color=color, linewidth=2)

ax.axhline(90, color='red', linestyle='--', linewidth=1.2, label='Target 90%')
ax.set_title('Adherence % Trend by Week and Campaign', fontweight='bold')
ax.set_xlabel('Week')
ax.set_ylabel('Avg Adherence %')
ax.set_ylim(80, 100)
ax.legend()
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 4. Analysis 2 — Shrinkage by Campaign

In [ ]:
shrinkage = (
    df.groupby(['campaign', 'shift'])
    .agg(shrinkage_pct=('shrinkage_minutes', lambda x: x.sum() / (df.loc[x.index, 'scheduled_minutes'].sum()) * 100))
    .reset_index()
)

fig, ax = plt.subplots()
sns.barplot(data=shrinkage, x='campaign', y='shrinkage_pct', hue='shift', ax=ax)
ax.axhline(15, color='red', linestyle='--', linewidth=1.2, label='Budget 15%')
ax.set_title('Shrinkage % by Campaign and Shift', fontweight='bold')
ax.set_xlabel('Campaign')
ax.set_ylabel('Shrinkage %')
ax.legend(title='Shift', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

## 5. Analysis 3 — Top 10 Agents by Productivity

In [ ]:
agent_stats = (
    active.groupby(['agent_id', 'campaign'])
    .agg(
        avg_contacts=('contacts_handled', 'mean'),
        avg_adherence=('adherence_pct', 'mean'),
        avg_qa=('qa_score', 'mean'),
        avg_csat=('csat_score', 'mean'),
    )
    .reset_index()
)
agent_stats['productivity_score'] = (
    agent_stats['avg_contacts'] * 0.35
    + agent_stats['avg_adherence'] * 0.25
    + agent_stats['avg_qa'] * 0.20
    + agent_stats['avg_csat'] * 0.20
).round(2)

top10 = agent_stats.nlargest(10, 'productivity_score').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [CAMPAIGN_COLORS[c] for c in top10['campaign']]
bars = ax.barh(top10['agent_id'], top10['productivity_score'], color=colors)
ax.set_title('Top 10 Agents by Productivity Score', fontweight='bold')
ax.set_xlabel('Productivity Score')
ax.invert_yaxis()

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=camp) for camp, c in CAMPAIGN_COLORS.items()]
ax.legend(handles=legend_elements, title='Campaign')
plt.tight_layout()
plt.show()

top10[['agent_id', 'campaign', 'productivity_score', 'avg_contacts', 'avg_adherence', 'avg_qa', 'avg_csat']].round(2)

## 6. Analysis 4 — Agents at Risk

In [ ]:
ADHERENCE_THRESHOLD = 85.0

agent_avg = (
    active.groupby(['agent_id', 'campaign'])['adherence_pct']
    .mean()
    .reset_index()
    .rename(columns={'adherence_pct': 'avg_adherence'})
)

at_risk = agent_avg[agent_avg['avg_adherence'] < ADHERENCE_THRESHOLD].sort_values('avg_adherence')

fig, ax = plt.subplots(figsize=(10, max(4, len(at_risk) * 0.3)))
colors = [CAMPAIGN_COLORS[c] for c in at_risk['campaign']]
ax.barh(at_risk['agent_id'], at_risk['avg_adherence'], color=colors, alpha=0.85)
ax.axvline(ADHERENCE_THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'Threshold {ADHERENCE_THRESHOLD}%')
ax.axvline(90, color='green', linestyle='--', linewidth=1.5, label='Target 90%')
ax.set_title(f'Agents at Risk — Avg Adherence < {ADHERENCE_THRESHOLD}%', fontweight='bold')
ax.set_xlabel('Avg Adherence %')
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
plt.show()

print(f'Agents at risk: {len(at_risk)} of {df["agent_id"].nunique()} ({len(at_risk)/df["agent_id"].nunique():.1%})')

## 7. Analysis 5 — Campaign SLA Performance

In [ ]:
sla = (
    active.groupby(['campaign', 'week_label'])
    .agg(sla_met_pct=('sla_met', lambda x: x.mean() * 100))
    .reset_index()
)

fig, ax = plt.subplots()
for campaign, color in CAMPAIGN_COLORS.items():
    sub = sla[sla['campaign'] == campaign]
    ax.plot(sub['week_label'], sub['sla_met_pct'], marker='s', label=campaign, color=color, linewidth=2)

ax.axhline(80, color='red', linestyle='--', linewidth=1.2, label='SLA Target 80%')
ax.set_title('SLA Compliance by Week and Campaign', fontweight='bold')
ax.set_xlabel('Week')
ax.set_ylabel('Days SLA Met (%)')
ax.set_ylim(50, 100)
ax.legend()
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 8. Analysis 6 — Occupancy, AHT, and CSAT Relationship

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Occupancy vs AHT
for campaign, color in CAMPAIGN_COLORS.items():
    sub = active[active['campaign'] == campaign]
    axes[0].scatter(sub['occupancy_pct'], sub['average_handle_time_sec'],
                    alpha=0.25, s=15, color=color, label=campaign)

# Regression line
x = active['occupancy_pct'].values
y = active['average_handle_time_sec'].values
m, b = np.polyfit(x, y, 1)
axes[0].plot(np.sort(x), m * np.sort(x) + b, 'k--', linewidth=1.5, label='Trend')
axes[0].set_xlabel('Occupancy %')
axes[0].set_ylabel('AHT (seconds)')
axes[0].set_title('Occupancy vs. AHT', fontweight='bold')
axes[0].legend(markerscale=2, fontsize=8)

# AHT vs CSAT
for campaign, color in CAMPAIGN_COLORS.items():
    sub = active[active['campaign'] == campaign]
    axes[1].scatter(sub['average_handle_time_sec'], sub['csat_score'],
                    alpha=0.25, s=15, color=color, label=campaign)

x2 = active['average_handle_time_sec'].values
y2 = active['csat_score'].dropna().values
x2_clean = active.loc[active['csat_score'].notna(), 'average_handle_time_sec'].values
m2, b2 = np.polyfit(x2_clean, y2, 1)
axes[1].plot(np.sort(x2_clean), m2 * np.sort(x2_clean) + b2, 'k--', linewidth=1.5, label='Trend')
axes[1].set_xlabel('AHT (seconds)')
axes[1].set_ylabel('CSAT Score')
axes[1].set_title('AHT vs. CSAT', fontweight='bold')
axes[1].legend(markerscale=2, fontsize=8)

plt.suptitle('Occupancy → AHT → CSAT Relationship', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Correlations
corr_occ_aht = active['occupancy_pct'].corr(active['average_handle_time_sec'])
corr_aht_csat = active['average_handle_time_sec'].corr(active['csat_score'])
corr_occ_csat = active['occupancy_pct'].corr(active['csat_score'])
print(f'Occupancy vs AHT:  r = {corr_occ_aht:+.3f}')
print(f'AHT vs CSAT:       r = {corr_aht_csat:+.3f}')
print(f'Occupancy vs CSAT: r = {corr_occ_csat:+.3f}')

## 9. Correlation Heatmap — All KPIs

In [ ]:
kpi_cols = [
    'adherence_pct', 'shrinkage_minutes', 'occupancy_pct',
    'contacts_handled', 'average_handle_time_sec',
    'qa_score', 'csat_score', 'service_level_pct',
    'late_login_minutes', 'overtime_minutes'
]
corr_matrix = active[kpi_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax
)
ax.set_title('KPI Correlation Matrix — All Active Records', fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Summary Statistics by Campaign

In [ ]:
summary = (
    active.groupby('campaign')
    .agg(
        agents=('agent_id', 'nunique'),
        avg_adherence=('adherence_pct', 'mean'),
        avg_shrinkage=('shrinkage_minutes', lambda x: x.sum() / active.loc[x.index, 'scheduled_minutes'].sum() * 100),
        avg_occupancy=('occupancy_pct', 'mean'),
        avg_contacts=('contacts_handled', 'mean'),
        avg_aht_sec=('average_handle_time_sec', 'mean'),
        avg_qa=('qa_score', 'mean'),
        avg_csat=('csat_score', 'mean'),
        sla_met_pct=('sla_met', lambda x: x.mean() * 100),
    )
    .round(2)
)
summary

---
## Key Takeaways

1. **Adherence is the leading indicator** — agents with adherence >90% consistently outperform on CSAT and contacts handled.
2. **Occupancy sweet spot is 80–88%** — above 90%, both AHT and CSAT degrade noticeably.
3. **Tech Support needs attention** — lowest SLA compliance AND highest AHT. Recommend staffing review.
4. **Customer Retention has the best QA/CSAT** — smaller team, more tenured agents. Model to replicate.
5. **Monday is the operational weak point** — highest absence and late-login rates. Buffer staffing recommended.

*See `reports/executive_summary.md` for full recommendations.*